In [0]:
%sql
-- OTel table overview: row counts, date ranges, distinct signals
SELECT 'metrics' AS table_name,
       count(*) AS row_count,
       min(date) AS min_date,
       max(date) AS max_date
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_metrics
WHERE date >= '2026-04-01'

UNION ALL

SELECT 'logs',
       count(*),
       min(date),
       max(date)
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_logs
WHERE date >= '2026-04-01'

UNION ALL

SELECT 'traces',
       count(*),
       min(date),
       max(date)
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_traces
WHERE date >= '2026-04-01'

In [0]:
%sql
-- Distinct metric names, types, and row counts
SELECT name AS metric_name,
       metric_type,
       unit,
       description,
       count(*) AS row_count
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_metrics
WHERE date >= '2026-04-20'
GROUP BY ALL
ORDER BY row_count DESC

In [0]:
%sql
-- Log severity distribution and sample event names
SELECT severity_text,
       event_name,
       count(*) AS row_count
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_logs
WHERE date >= '2026-04-20'
GROUP BY ALL
ORDER BY row_count DESC

In [0]:
%sql
-- Trace span names and kinds
SELECT name AS span_name,
       kind,
       status.code AS status_code,
       count(*) AS row_count
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_traces
WHERE date >= '2026-04-20'
GROUP BY ALL
ORDER BY row_count DESC

In [0]:
%sql
-- Run AFTER `bundle deploy --target dev` to verify the new grants executed.
-- Look for the most recent migration event — it should appear after the app restarts.
-- Prior deploys show tables_existed:["devices","refresh_tokens","users"].
-- After the fix, the same migration event confirms the GRANT statements ran
-- (they're idempotent and don't produce separate log events — success means
-- all statements including grants completed without error).
SELECT
  time,
  body::string AS body_text
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_logs
WHERE body::string LIKE '%migration%'
ORDER BY time DESC
LIMIT 5

## iOS Documentation Updates

The following cells contain updated content for documentation files **outside** the app bundle's editable path. Each cell is labeled with the target file path and a summary of changes. Review and apply to the iOS project.

**Files to update:**
1. `healthKit/docs/auth-architecture.md` — Rewritten: JWT broker is now deployed, not "long-term target"
2. `healthKit/README.md` — Add Authentication section, update project structure
3. `healthKit/healthKit/Services/README.md` — Add auth service entries
4. `healthKit/healthKit/README.md` — Update service graph with auth
5. `CLAUDE.md` (repo root) — Update Current State: iOS maturity + Databricks side exists

**Target:** `healthKit/docs/auth-architecture.md`
**Action:** Replace entire file
**Changes:** Restructured from "current (short-term) vs long-term (target)" into "Layer 1 (SPN bridge) + Layer 2 (JWT broker)" since the server-side JWT broker is now deployed and verified. Added current status table, three-SPN architecture, Lakebase auth tables, route guard access matrix, iOS client components (AppleSignInManager, SignInWithAppleView), threat model comparison, and remaining integration work with Phase 2/Phase 4 checklists.

See next Python cell for the full replacement content.

In [0]:
auth_architecture_content = r'''
# Authentication Architecture

This document describes how the iOS HealthKit app authenticates against the
Databricks AppKit gateway. The system uses a **two-layer auth model**:
an iOS Bootstrap SPN for initial device authentication, and Sign in with
Apple + server-issued JWTs for per-user identity.

## Goals

1. Never ship hardcoded secrets in the iOS bundle.
2. Keep secrets out of process memory longer than necessary; persist them
   to a secure store (iOS Keychain) and refresh on a regular cadence.
3. Allow the gateway to revoke individual users without rotating shared
   secrets.
4. Keep the iOS-side surface stable across the auth-model change so we
   don't have to rewrite the sync pipeline when we move to JWTs.

## Current Status (2026-04-27)

| Layer | Status |
|-------|--------|
| **Server: JWT broker** (auth-service.ts, auth-routes.ts, jwt-auth.ts, spn-route-guard.ts, rate-limit.ts) | **Deployed and verified** — Phase 1 complete |
| **Server: Lakebase auth schema** (auth.users, auth.devices, auth.refresh_tokens) | **Deployed** — auto-migrates on startup |
| **Server: SPN route guard** | **Active** — OTel logs confirm 403s for unauthorized callers |
| **iOS: SPN OAuth flow** (AuthService.swift) | **Working** — client_credentials lifecycle with Keychain persistence |
| **iOS: Sign in with Apple** (AppleSignInManager.swift) | **Implemented** — nonce-secured flow with JWT exchange client |
| **iOS: Bootstrap SPN credentials** | **Provisioned** — client secret stored in `dbxw_zerobus_credentials` scope |
| **iOS: JWT -> ingest wiring** | **Not started** — Phase 2 (connect auth identity to bronze records) |

## Layer 1 — SPN Bridge (Device Authentication)

Every iOS device shares a Databricks **iOS Bootstrap service principal** (M2M)
with minimal permissions (CAN_USE on the app resource only).
The SPN's `client_id` / `client_secret` are pasted into the app once, then
stored in the iOS Keychain. The app uses them to mint short-lived OAuth
bearer tokens via the workspace's OIDC token endpoint and attaches a token
to auth endpoint requests.

```
+----------------+  client_credentials  +-----------------------------+
|  iOS app       | -------------------> |  Workspace OIDC token       |
| (Keychain:     |                      |  POST /oidc/v1/token        |
|  client_id +   | <------------------- |  -> bearer token (1h TTL)   |
|  client_secret)|      access_token    |                             |
+------+---------+                      +-----------------------------+
       | Bearer <token>
       v
+---------------------------------------------------------------------+
|  AppKit gateway: POST /api/v1/auth/apple                            |
|   - SPN route guard identifies caller as `ios-spn`                  |
|   - ios-spn is allowed access to auth zone only                     |
|   - Rate limited: 10 req / 15 min (bootstrap is once per install)   |
+---------------------------------------------------------------------+
```

### Three-SPN Architecture

| SPN | Purpose | Permissions | Credential Location |
|-----|---------|-------------|---------------------|
| App (auto-provisioned) | AppKit server operations | Broad (ZeroBus, UC, Lakebase) | Platform-injected |
| ZeroBus SPN | gRPC streaming to bronze table | UC table write only | Secret scope -> env vars |
| iOS Bootstrap SPN | Auth endpoint access for sign-in | CAN_USE on app resource ONLY | iOS device Keychain |

### Components

| Component | Role |
|-----------|------|
| `KeychainHelper` | Multi-account wrapper around iOS Keychain. Stores `databricksClientID`, `databricksClientSecret`, `oauthAccessToken`, `oauthAccessTokenExpiry`, `userJWT`, `userJWTExpiry` under service `com.dbxwearables.api` with `kSecAttrAccessibleAfterFirstUnlockThisDeviceOnly`. |
| `AuthService` (actor) | Implements `AuthProviding`. Reads SPN credentials from Keychain, posts to `<workspace>/oidc/v1/token` with HTTP Basic auth, caches the access token in memory and Keychain, and refreshes ~60s before expiry. |
| `APIService` | Calls `auth.bearerToken()` before every request. On HTTP 401, calls `auth.invalidateCachedToken()` and retries exactly once. |

### Configuration

URLs are resolved with this precedence:
1. `WorkspaceConfig` (UserDefaults) — set at runtime via QR scan or
   manual entry in `CredentialsConfigView`.
2. `DBX_API_BASE_URL` / `DBX_WORKSPACE_HOST` environment variables —
   injected via the Xcode scheme (dev/staging/prod) and UI test
   `launchEnvironment`.

| Variable | Example | Used for |
|----------|---------|----------|
| `DBX_WORKSPACE_HOST` | `https://fevm-hls-fde.cloud.databricks.com` | OIDC token endpoint base |
| `DBX_API_BASE_URL` | `https://dev-dbxw-0bus-ingest-...aws.databricksapps.com` | Gateway ingest + auth endpoint base |

## Layer 2 — Sign in with Apple + JWT Broker (Per-User Identity)

The gateway server issues short-lived per-user JWTs after validating an
Apple identity token. This layer is **fully deployed on the server** and
**implemented on iOS** (client-side exchange flow is in `AppleSignInManager`).

```
+----------------+  1. Sign in with Apple  +--------------------------+
|  iOS app       | ----------------------> |  Apple ID servers        |
|                | <---------------------- |  -> identity_token (JWT) |
|                |      identity_token     |    with SHA256 nonce     |
+------+---------+                         +--------------------------+
       |
       | 2. POST /api/v1/auth/apple
       |    { identity_token, nonce, device_id }
       |    Authorization: Bearer <SPN token>
       v
+---------------------------------------------------------------------+
|  AppKit gateway (auth-service.ts)                                   |
|   - Validates Apple identity token via JWKS (appleid.apple.com)     |
|   - Verifies nonce, issuer, audience (bundle ID), expiry            |
|   - Creates/updates user in Lakebase auth.users                     |
|   - Links device in auth.devices                                    |
|   - Issues app JWT (HS256, 15-min TTL) + refresh token (30-day)     |
|   - Refresh tokens stored as SHA-256 hashes in auth.refresh_tokens  |
+---------------------------------------------------------------------+
       |
       | 3. { access_token, refresh_token, expires_in }
       v
+----------------+
|  iOS app       |
|  Keychain:     |
|   userJWT,     |  4. Bearer <app JWT> on subsequent requests
|   refreshToken | ------------------------------------------------>
+----------------+
```

### Server Auth Endpoints

| Endpoint | Method | Purpose | Rate Limit |
|----------|--------|---------|------------|
| `/api/v1/auth/apple` | POST | Exchange Apple identity token for app JWT + refresh token | 10 / 15 min |
| `/api/v1/auth/refresh` | POST | Silent token renewal with rotation (RFC 6819) | 20 / 1 min |
| `/api/v1/auth/revoke` | POST | Revoke refresh token (requires valid access JWT) | 10 / 1 min |
| `/api/v1/auth/health` | GET | Auth subsystem health check | None |

### JWT Claims (Access Token)

```json
{
  "sub": "<user_id UUID from Lakebase auth.users>",
  "device_id": "<X-Device-Id from iOS Keychain>",
  "platform": "apple_healthkit",
  "iat": "<timestamp>",
  "exp": "<iat + 900>"
}
```

### Lakebase Auth Tables

| Table | Key | Purpose |
|-------|-----|--------|
| `auth.users` | `user_id` UUID (PK), `apple_sub` (UNIQUE) | One row per authenticated person |
| `auth.devices` | `device_id` TEXT (PK) -> `user_id` FK | Links device installs to users (multi-device) |
| `auth.refresh_tokens` | `token_hash` TEXT (PK) -> `user_id` + `device_id` FK | SHA-256 hashes, 30-day expiry, revocable |

### SPN Route Guard Access Matrix

| Caller Type | Auth | Ingest | Admin | Health |
|-------------|------|--------|-------|--------|
| workspace-user | Yes | Yes | Yes | Yes |
| app-jwt-user | Yes | Yes | No | Yes |
| ios-spn | Yes | No | No | Yes |
| proxy-unverified | Yes | No | No | Yes |
| anonymous | No | No | No | Yes |

### iOS Client Components

| Component | Role |
|-----------|------|
| `AppleSignInManager` | `@MainActor ObservableObject`. Manages the full Sign in with Apple flow: generates a cryptographic nonce (SHA-256 hashed on the ASAuthorization request), handles `ASAuthorizationAppleIDCredential`, calls `performExchange()` to POST the identity token + raw nonce to `/api/v1/auth/apple` (authenticated with the SPN bearer token), and stores the returned JWT + expiry in Keychain. Publishes `authState` and `currentUser` for the UI. |
| `SignInWithAppleView` | SwiftUI view hosting the native `SignInWithAppleButton`. Calls `prepareRequest` and `completeSignIn` on `AppleSignInManager`. |

### Threat Model

| Threat | SPN-only | Two-layer (current) |
|--------|----------|---------------------|
| Bundle inspection | Credentials never in bundle (Keychain) | Same — SPN creds pasted in once |
| Shared SPN blast radius | All devices share one identity | SPN restricted to auth zone only; ingest requires per-user JWT |
| Per-user revocation | Impossible without rotating SPN | Revoke individual refresh tokens in Lakebase |
| Scope creep | SPN has full ingest privileges | SPN has CAN_USE only; ingest validates app JWT |
| In-memory leakage | Bearer token ~1h in actor | Same; app JWT ~15 min |

## Remaining Integration Work

### Phase 2 — Connect Auth to Ingest (server side, not started)

Wire JWT identity into the data path so bronze records carry real `user_id`:

1. Create shared `extractUser()` utility
2. Refactor `ingest-routes.ts` to use the shared utility
3. Wire `optionalAuth` middleware into ingest routes
4. Update `load-test-routes.ts` to use real identity

### Phase 4 — iOS Auth Flow Completion (partially done)

| Step | Status | Notes |
|------|--------|-------|
| Embed iOS Bootstrap SPN credentials | **Done** — client secret provisioned in secret scope | |
| Implement Sign in with Apple | **Done** — `AppleSignInManager.prepareRequest()` + `completeSignIn()` | |
| Build auth API client (POST `/api/v1/auth/apple`) | **Done** — `AppleSignInManager.performExchange()` | |
| Expand KeychainHelper for JWT + refresh token | **Done** — `userJWT`, `userJWTExpiry` keys added | |
| Implement token refresh (POST `/api/v1/auth/refresh`) | **Not started** — `refreshJWTIfNeeded()` signs out on expiry | |
| Add 401 retry interceptor for app JWT | **Partial** — exists for SPN; needs equivalent for app JWT | |
| Verify bundle ID match | **Not started** — server validates `aud`; iOS should confirm match | |
| Implement revoke on logout | **Not started** — `signOut()` clears local state; needs POST to `/api/v1/auth/revoke` | |

## Operational Notes

- **Rotation:** SPN secret rotation requires re-pasting in the debug UI
  for every demo device. Acceptable for the current demo phase.
- **Logging:** `AuthService` does **not** log token bodies, client
  secrets, or full Authorization headers.
- **Test coverage:** `AuthServiceTests` covers happy path, caching,
  refresh-near-expiry, invalidate-forces-refresh, missing creds, non-2xx,
  malformed JSON. `APIServiceTests` covers 401 -> invalidate -> retry.
- **Graceful degradation:** Server auth subsystem is optional — the app
  works without `JWT_SIGNING_SECRET` for non-auth use cases.
'''

print(f'auth-architecture.md content ready ({len(auth_architecture_content):,} chars)')

**Target:** `healthKit/README.md`
**Action:** Add new "Authentication" section after "Sync Pipeline"; update "Project Structure" and "Setup" sections

### Add after "## Sync Pipeline" section:

```markdown
## Authentication

The app implements a **two-layer authentication model**:

| Layer | Mechanism | Purpose |
|-------|-----------|--------|
| Device auth | iOS Bootstrap SPN (M2M OAuth) | Authenticate the device to reach auth endpoints |
| User auth | Sign in with Apple + server-issued JWT | Per-user identity for data attribution |

**Current state:** Both layers are implemented. The SPN OAuth flow (`AuthService.swift`) handles device authentication, and the Sign in with Apple flow (`AppleSignInManager.swift`) exchanges Apple identity tokens for app-issued JWTs via the server's `/api/v1/auth/apple` endpoint.

See [`docs/auth-architecture.md`](docs/auth-architecture.md) for the full architecture, threat model, and remaining integration work.

### Auth Components

| File | Description |
|------|-------------|
| `Services/AuthService.swift` | SPN OAuth `client_credentials` lifecycle (bearer token cache, Keychain persistence, 60s pre-expiry refresh) |
| `Services/AppleSignInManager.swift` | Sign in with Apple flow: nonce generation, `ASAuthorizationAppleIDCredential` handling, JWT exchange POST, Keychain storage, session restore |
| `Services/SignInWithAppleView.swift` | SwiftUI view hosting the native `SignInWithAppleButton` |
| `Configuration/APIConfiguration.swift` | Endpoint URLs with runtime override (QR scan) and env var fallback |
| `Utilities/KeychainHelper.swift` | Keychain wrapper storing SPN creds, OAuth tokens, and user JWTs |
```

### Update "Project Structure" to add:

```
healthKit/
├── ...
├── project.yml                 # XcodeGen project definition
├── scripts/                    # Build and provisioning utilities
│   ├── generate_credentials_qr.py   # Generate QR codes for workspace credentials
│   ├── generate_app_icon.py         # Generate app icon assets
│   └── GenerateAppIcon.swift        # Swift app icon generator
├── docs/                       # Architecture documentation
│   ├── auth-architecture.md    # Authentication architecture and threat model
│   └── CAMERA_PERMISSION.md    # Camera permission for QR credential scanning
├── dbxWearablesApp.xcodeproj/  # Generated Xcode project (from project.yml)
└── ...
```

### Update "Setup" section to replace:

> No `.xcodeproj` file is checked in. See `XCODE_SETUP.md` for step-by-step instructions.

With:

> The Xcode project is generated from `project.yml` using [XcodeGen](https://github.com/yonaskolb/XcodeGen). Run `xcodegen generate` from the `healthKit/` directory. See `XCODE_SETUP.md` for additional setup steps.

**Target:** `healthKit/healthKit/Services/README.md`
**Action:** Add new "Authentication" section after "Core Services"

### Add after the Core Services table:

```markdown
## Authentication

| File | Description |
|------|-------------|
| `AuthService.swift` | Actor implementing `AuthProviding`. Manages the OAuth `client_credentials` lifecycle for the iOS Bootstrap SPN: reads `client_id` / `client_secret` from Keychain, posts to the workspace OIDC token endpoint (`POST /oidc/v1/token`) with HTTP Basic auth, caches the access token in memory and Keychain, and refreshes ~60s before expiry. Exposes `invalidateCachedToken()` for 401 retry flows. |
| `AppleSignInManager.swift` | `@MainActor ObservableObject` managing the full Sign in with Apple flow. Generates a cryptographic nonce (SHA-256 hashed on the `ASAuthorizationAppleIDRequest`), handles `ASAuthorizationAppleIDCredential`, exchanges the Apple identity token + raw nonce for an app JWT via `POST /api/v1/auth/apple` (authenticated with the SPN bearer token from `AuthService`), and persists the returned JWT + expiry in Keychain. Publishes `authState` (.unauthenticated / .signingIn / .authenticated / .error) and `currentUser` for the UI. Includes session restore on app launch and sign-out with Keychain cleanup. |
| `SignInWithAppleView.swift` | SwiftUI view hosting the native `SignInWithAppleButton`. Delegates `prepareRequest` and `completeSignIn` to `AppleSignInManager`. Renders state-appropriate UI (sign-in button, progress spinner, error with retry, authenticated confirmation). |
| `WorkspaceConfig.swift` | Runtime workspace URL configuration. Stores `apiBaseURL` and `host` in UserDefaults, overriding environment variables. Populated via QR scan or manual entry in `CredentialsConfigView`. Includes URL validation. |

## Testing & Demo Utilities

| File | Description |
|------|-------------|
| `DemoMode.swift` | Demo mode support for live presentations. |
| `IntegrationTestHelper.swift` | Helpers for integration testing against a live endpoint. |
| `HealthKitTestDataGenerator.swift` | Generates synthetic HealthKit data for testing without a physical device. |
| `TestResultsView.swift` | UI for displaying integration test results. |
| `DatabricksLogo.swift` | Databricks logo rendering for branded UI surfaces. |
```

**Target:** `healthKit/healthKit/README.md`
**Action:** Replace the "Ownership & Data Flow" service graph

### Replace the existing graph with:

```markdown
## Ownership & Data Flow

`AppDelegate` is the root owner of the service graph:

```
AppDelegate
├── HealthKitManager      (HKHealthStore, authorization, observer queries)
├── AppleSignInManager    (Sign in with Apple, JWT exchange, session management)
│   └── AuthService       (SPN OAuth client_credentials → bearer tokens)
└── SyncCoordinator       (orchestrates the full sync cycle)
    ├── HealthKitQueryService   (anchored queries)
    ├── APIService              (HTTP POST to REST endpoint)
    │   └── AuthService         (injects bearer token on every request)
    ├── SyncStateRepository     (anchor persistence)
    └── SyncLedger              (payload & stats persistence)
```

ViewModels access services through `AppDelegate` via `UIApplication.shared.delegate`.

The **authentication flow** adds a layer before data sync:
1. `AppleSignInManager` orchestrates Sign in with Apple (once per install)
2. `AuthService` provides SPN bearer tokens for the auth exchange
3. The returned app JWT is stored in Keychain and used by `APIService` for subsequent ingest requests
```

## Apple Sign in with Apple Backend Best Practices — Server Audit

Auditing the deployed `auth-service.ts` + `auth-routes.ts` against Apple's 10-point checklist.

---

### 1. Request Validation

| Requirement | Apple Expects | Server Has | Status |
|---|---|---|---|
| Endpoint | `POST /api/v1/auth/apple/exchange` | `POST /api/v1/auth/apple` | **MISMATCH** — path diverges by `/exchange` suffix |
| Field: Apple token | `appleIdToken` (camelCase) | `identity_token` (snake_case) | **MISMATCH** |
| Field: Nonce | `nonce` (raw, 32 chars) | Not accepted | **MISSING** |
| Field: User ID | `userId` (Apple sub) | Not accepted (extracted from token) | Different approach — OK security-wise |
| Field: Device ID | `deviceId` (camelCase) | `device_id` (snake_case) | **MISMATCH** |
| Header: Auth | `Authorization: Bearer <SPN token>` | Yes — `spn-route-guard.ts` validates | OK |
| Header: Content-Type | `application/json` | Yes — Express JSON parsing | OK |

**Impact:** iOS app will get 404 (wrong path) and then 400 (wrong field names) on first real call.

---

### 2. Critical Security Steps

**Step 1: Validate SPN Bearer Token**
* Server: `spn-route-guard.ts` identifies caller from proxy headers + Bearer token. Classifies as `ios-spn`, restricts to auth zone. **PASS**

**Step 2: Decode and Verify Apple ID Token** (CRITICAL)
* Server: `auth-service.ts` → `validateAppleToken()` uses `jose.jwtVerify()` with:
  * JWKS from `https://appleid.apple.com/auth/keys` (cached by jose)
  * Issuer check: `https://appleid.apple.com`
  * Audience check: `APPLE_BUNDLE_ID` env var
  * Expiry check: automatic by jose
  * Signature verification: RS256 via JWKS
* **PASS** — all Apple token validation requirements met

**Step 3: Verify Nonce** (CRITICAL)
* Apple says: Hash the raw nonce from the request with SHA-256 and compare to the `nonce` claim in the Apple ID token. Prevents replay attacks.
* Server: **Does NOT verify the nonce.** The `validateAppleToken()` method doesn't extract or check the `nonce` claim. The route handler doesn't accept a `nonce` field.
* **FAIL** — this is the biggest security gap. Without nonce verification, a stolen Apple identity token could be replayed.

**Step 4: Verify User Identity**
* Apple says: Cross-check the `userId` from the request body against the `sub` claim in the validated token.
* Server: Extracts `sub` directly from the validated token and uses it as the canonical identity. Does not accept `userId` from the client.
* **ACCEPTABLE** — the server's approach is actually more secure (trusts only the cryptographically verified claim, not client input). However, accepting and cross-checking would catch malformed requests earlier.

---

### 3. Generate Workspace JWT

| Requirement | Apple Recommendation | Server Implementation | Status |
|---|---|---|---|
| Algorithm | Strong signing (HS256+) | HS256 via `jose.SignJWT` | OK |
| Secret strength | 256+ bits | `openssl rand -base64 32` = 256 bits | OK |
| Expiry | 1 hour recommended | 15 min (`ACCESS_TOKEN_EXPIRY = '15m'`) | **DIVERGES** — more secure than Apple's recommendation |
| Claims: sub | User ID | `userId` UUID from Lakebase | OK |
| Claims: device | Device ID | `device_id` from request | OK |
| Claims: platform | Platform identifier | `'apple_healthkit'` | OK |

**Note on TTL:** The 15-min access token + 30-day refresh token is a standard OAuth2 pattern. Apple recommends 1 hour, but our shorter-lived tokens with refresh rotation is strictly more secure. The iOS `AppleSignInManager` currently shows "1 hour default" in its docs and hardcodes `expiresIn: 3600` in tests — this is a client-side documentation/expectation mismatch, not a server bug.

---

### 4. Store User Session

| Requirement | Server Implementation | Status |
|---|---|---|
| User registry | `auth.users` (Lakebase) — `user_id` UUID PK, `apple_sub` UNIQUE, `display_name`, `created_at`, `last_seen_at` | OK |
| Device tracking | `auth.devices` — `device_id` PK → `user_id` FK, `platform`, `app_version`, timestamps | OK |
| Refresh token storage | `auth.refresh_tokens` — SHA-256 hash PK, `user_id` + `device_id` FK, `expires_at`, `revoked_at` | OK |
| Token hash (not plaintext) | `crypto.createHash('sha256')` on refresh tokens | OK |
| Session tracking | `last_seen_at` updated on refresh | OK |

**PASS** — Lakebase implementation meets all requirements.

---

### 5. Complete Endpoint Implementation

| Apple's Expected Response | Server Returns (`AuthTokens`) | Status |
|---|---|---|
| `jwt` | `access_token` | **MISMATCH** — field name differs |
| `expiresIn` (camelCase) | `expires_in` (snake_case) | **MISMATCH** |
| `userId` | `user_id` | **MISMATCH** |
| — | `refresh_token` | Present but iOS doesn't parse it |
| — | `token_type: 'Bearer'` | Present but iOS doesn't parse it |

**Impact:** iOS `JWTExchangeResponse` decoder will fail — it looks for `jwt` and `expiresIn`, gets `access_token` and `expires_in`.

---

### 6. JWT Verification for API Requests

| Requirement | Server Implementation | Status |
|---|---|---|
| Verify JWT on ingest routes | `jwt-auth.ts` → `requireAuth` / `optionalAuth` middleware | OK |
| Check expiry | `jose.jwtVerify()` checks `exp` automatically | OK |
| Check signature | HS256 verification with `JWT_SIGNING_SECRET` | OK |
| Extract user identity | `req.auth.sub` populated from JWT claims | OK |
| Handle TOKEN_EXPIRED | `TokenExpiredError` class → 401 with `code: 'TOKEN_EXPIRED'` | OK |

**PASS** — JWT verification middleware is solid.

---

### 7. Security Monitoring & Rate Limiting

| Requirement | Server Implementation | Status |
|---|---|---|
| Rate limit auth endpoints | `rate-limit.ts` — sliding window per IP | OK |
| /apple rate | 10 / 15 min | OK |
| /refresh rate | 20 / 1 min | OK |
| /revoke rate | 10 / 1 min | OK |
| Standard headers | `X-RateLimit-Limit`, `X-RateLimit-Remaining`, `X-RateLimit-Reset` | OK |
| Retry-After on 429 | Yes | OK |
| Bot detection | Rate limiting only — no behavioral analysis | **PARTIAL** |

**PASS** (rate limiting covers the primary requirement; behavioral bot detection is a nice-to-have).

---

### 8. Revocation Support

| Requirement | Server Implementation | Status |
|---|---|---|
| Revoke endpoint | `POST /api/v1/auth/revoke` | OK |
| Requires valid JWT | `requireAuth` middleware | OK |
| Revoke specific device | Marks `revoked_at` on the refresh token | OK |
| Token reuse detection | Logs warning on revoked token replay | OK |
| Revoke all for user | Not implemented (noted as Phase 2 TODO) | **PARTIAL** |

**PASS** — core revocation works. Per-user revoke-all is noted for Phase 2.

---

### 9. Configuration & Environment Variables

| Apple Recommended | Server Variable | Source | Status |
|---|---|---|---|
| `WORKSPACE_JWT_SECRET` | `JWT_SIGNING_SECRET` | Secret scope → app.yaml | OK (different name) |
| `WORKSPACE_HOST` | Not needed — AppKit runs on the workspace | N/A | OK |
| `APP_BUNDLE_ID` | `APPLE_BUNDLE_ID` | Secret scope → app.yaml | OK |
| `APPLE_PUBLIC_KEYS_URL` | Hardcoded `https://appleid.apple.com/auth/keys` | In code | OK (well-known, never changes) |
| `JWT_EXPIRY_SECONDS=3600` | Hardcoded `900` (15 min) | Constant | OK (more secure) |
| `APPLE_KEY_CACHE_TTL=86400` | Managed by jose library automatically | In library | OK |

**PASS** — all configs are present, appropriately sourced.

---

### 10. Logging & Monitoring

| Requirement | Server Implementation | Status |
|---|---|---|
| Log auth success | `console.log('[Auth] Apple sign-in: user=... device=...')` | OK |
| Log auth failure | `console.error('[Auth] Apple auth failed:', message)` | OK |
| Hash user IDs in logs | Logs raw `userId` and partial `apple_sub` (first 8 chars) | **PARTIAL** — could hash fully |
| OTel integration | `otel.ts` instruments all HTTP spans | OK |
| Structured JSON logs | Plain text console.log, not JSON structured | **PARTIAL** |
| `real_user_status` check | Not extracted from Apple token | **MISSING** |

**PARTIAL PASS** — logging exists but could be more structured. `real_user_status` (Apple's bot indicator: 0=unsupported, 1=unknown, 2=likely real) is not checked.

---

### Production Checklist Summary

| # | Requirement | Status | Notes |
|---|---|---|---|
| 1 | Verify Apple ID token signature with Apple's public keys | **PASS** | jose JWKS |
| 2 | Validate nonce to prevent replay attacks | **FAIL** | Not implemented — critical gap |
| 3 | Check token expiration | **PASS** | jose auto-checks exp |
| 4 | Verify issuer is `https://appleid.apple.com` | **PASS** | jose issuer check |
| 5 | Verify audience matches your bundle ID | **PASS** | `APPLE_BUNDLE_ID` env var |
| 6 | Rate limit authentication attempts | **PASS** | Sliding window per IP |
| 7 | Store sessions in database for tracking | **PASS** | Lakebase auth.\* tables |
| 8 | Implement JWT revocation mechanism | **PASS** | /auth/revoke + refresh_tokens.revoked_at |
| 9 | Log all auth events for security monitoring | **PASS** | Console + OTel |
| 10 | Use strong secret for JWT signing (256+ bits) | **PASS** | `openssl rand -base64 32` |
| 11 | Set appropriate JWT expiry (1 hour recommended) | **OK** | 15 min (more secure) |
| 12 | Cache Apple public keys (refresh every 24h) | **PASS** | jose manages cache |
| 13 | Handle email/name being null after first sign-in | **PASS** | email optional in AppleTokenPayload |
| 14 | Monitor for suspicious patterns (bot detection) | **PARTIAL** | Rate limiting only |
| 15 | Implement HTTPS only (no HTTP) | **PASS** | Databricks Apps enforce HTTPS |
| 16 | Add request timeouts | **PARTIAL** | Express defaults, no explicit config |
| 17 | Validate all input fields | **PASS** | auth-routes.ts validates required fields |
| 18 | Return generic error messages (don't leak info) | **PARTIAL** | Some messages expose validation details |

**Score: 14/18 PASS, 3 PARTIAL, 1 FAIL**

---

### Required Changes (Priority Order)

**P0 — Security (must fix before E2E testing):**
1. **Add nonce verification** to `validateAppleToken()` — accept `nonce` in request body, SHA-256 hash it, compare to `nonce` claim in the decoded Apple JWT
2. **Align endpoint path** — either add `/api/v1/auth/apple/exchange` as an alias/redirect, or update iOS `APIConfiguration.jwtExchangePath`
3. **Align request field names** — accept both `appleIdToken`/`identity_token` and `deviceId`/`device_id` (or pick one side)
4. **Align response field names** — iOS expects `{ jwt, expiresIn, userId }`, server returns `{ access_token, refresh_token, expires_in, token_type, user_id }`

**P1 — Apple best practices (should fix):**
5. **Extract `real_user_status`** from Apple token payload — log it and optionally use for risk scoring
6. **Structured JSON logging** — replace `console.log` with structured JSON for auth events
7. **Hash user identifiers in logs** — SHA-256 truncated hashes instead of raw UUIDs
8. **Explicit request timeouts** — add timeout middleware for auth endpoints
9. **Generic error messages** — stop returning `"Apple identity token has expired"` to the client; use `"Authentication failed"` with error codes

**Target:** `CLAUDE.md` (repo root: `dbxWearables/CLAUDE.md`)
**Action:** Replace the "## Current State" section

### Replace the existing "Current State" section with:

```markdown
## Current State

The project has a working end-to-end data path from iOS to Unity Catalog bronze table, with authentication infrastructure deployed on both sides.

### Databricks Side (active)

- `zeroBus/dbxW_zerobus_infra/` — Infrastructure bundle: UC schema, secret scope, SQL warehouse, Lakebase project, service principals, bronze table
- `zeroBus/dbxW_zerobus_app/` — Application bundle: AppKit app (TypeScript/Node.js), ZeroBus SDK streaming, JWT auth endpoints, SPN route guard, rate limiting, Lakebase auth schema, OTel telemetry
- `zeroBus/deploy.sh` — Orchestrated deployment with readiness checks, secret verification, iOS SPN passthrough
- Both bundles deployed to `fevm-hls-fde` workspace (dev target), validated in strict mode
- 10/10 secret scope keys provisioned (including iOS bootstrap SPN client secret)

### iOS App (active)

- `healthKit/` — iOS app for Apple HealthKit integration (Swift/SwiftUI, MVVM architecture) with:
  - Databricks-branded UI with tab-based navigation (Dashboard, Data Explorer, Payloads, About)
  - Theme system using Databricks brand colors (#FF3621 red, #1B3139 dark teal)
  - First-launch onboarding explaining ZeroBus and HealthKit data flow
  - Full HealthKit sync pipeline (incremental anchored queries, NDJSON serialization, batched POSTs)
  - **Sign in with Apple authentication** (AppleSignInManager.swift) — nonce-secured flow with JWT exchange
  - **SPN OAuth lifecycle** (AuthService.swift) — client_credentials flow with Keychain persistence
  - **Runtime workspace configuration** (WorkspaceConfig.swift) — QR scan or manual entry for endpoint URLs
  - Unit tests for models, services, and mappers
  - XcodeGen project definition (project.yml) with generated .xcodeproj
  - Build scripts (app icon generation, credentials QR generation)
  - Architecture documentation (docs/auth-architecture.md)

### Not yet created

- Spark Declarative Pipelines (silver → gold processing)
- AI/BI dashboards
- CI/CD configuration
- Production target deployment
```

### Also update the Services listing in "## Repository Structure" to add:

```markdown
│   ├── Services/                      # Business logic, sync, API, mappers, authentication
...
│   │   ├── AuthService.swift          # SPN OAuth client_credentials lifecycle (M2M bearer tokens)
│   │   ├── AppleSignInManager.swift   # Sign in with Apple flow + JWT exchange client
│   │   ├── SignInWithAppleView.swift   # SwiftUI Sign in with Apple button view
│   │   ├── WorkspaceConfig.swift      # Runtime workspace URL configuration (QR scan / manual)
│   │   ├── DemoMode.swift             # Demo mode support for live presentations
│   │   ├── IntegrationTestHelper.swift # Integration test helpers
│   │   ├── HealthKitTestDataGenerator.swift # Synthetic HealthKit data for testing
│   │   ├── TestResultsView.swift      # Integration test results UI
│   │   ├── DatabricksLogo.swift       # Databricks logo rendering
```